In [154]:
import pandas as pd
import scipy
import data_profiling
from data_profiling import ProfileReport
import numpy as np

In [155]:
uk_prices_2026 = pd.read_csv("UK_elec_data_1y_Oct2026.csv", sep=";")
uk_prices_hist = pd.read_csv("United_Kingdom.csv")
exchange_rates = pd.read_csv("EURO_GBP_rates.csv")
day_ahead_demand = pd.read_csv("dayahead_demand.csv") # check if always before 09:20
day_ahead_wind = pd.read_csv("dayahead_wind.csv") # check if always before 09:20

In [156]:
uk_prices_hist.head() 

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe)
0,United Kingdom,GBR,2016-06-29 23:00:00,2016-06-30 00:00:00,35.72
1,United Kingdom,GBR,2016-06-30 00:00:00,2016-06-30 01:00:00,35.49
2,United Kingdom,GBR,2016-06-30 01:00:00,2016-06-30 02:00:00,35.07
3,United Kingdom,GBR,2016-06-30 02:00:00,2016-06-30 03:00:00,31.50
4,United Kingdom,GBR,2016-06-30 03:00:00,2016-06-30 04:00:00,30.27


In [157]:
ukp = uk_prices_hist[["Datetime (Local)", "Price (EUR/MWhe)"]]
ukp["Datetime (Local)"] = pd.to_datetime(ukp["Datetime (Local)"])
ProfileReport(ukp)
;

C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\1065166300.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ukp["Datetime (Local)"] = pd.to_datetime(ukp["Datetime (Local)"])


''

In [158]:
# converting to GBP

old_name = exchange_rates.columns.to_list()[-1]
exchange_rates = exchange_rates.rename(columns={old_name: "exchange_rate"})
exchange_rates["Date"] = pd.to_datetime(exchange_rates["Date"], format="%d %b %y")
exchange_rates.head()

,Date,exchange_rate
0,2026-10-27,1.1946
1,2026-10-26,1.1940
2,2026-10-23,1.1947
3,2026-10-22,1.1934
4,2026-10-21,1.1924


In [159]:
# filling in weekend exchange rates with the exchange rate on Friday 

full_range = pd.date_range(exchange_rates["Date"].min(), exchange_rates["Date"].max(), freq="D")

exchange_rates = (
    exchange_rates.set_index("Date")
        .sort_index()
        .reindex(full_range)
        .ffill()
        .rename_axis("Date")
        .reset_index()
)

In [160]:
ukp["Date"] = ukp["Datetime (Local)"].dt.normalize()

# Merge the daily rate onto each row
ukp = ukp.merge(
    exchange_rates[["Date", "exchange_rate"]],
    on="Date",
    how="left",
)

# EUR/MWhe * GBP/EUR = GBP/MWhe
ukp["Price (GBP/MWhe)"] = ukp["Price (EUR/MWhe)"] / ukp["exchange_rate"]

C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\2129027785.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ukp["Date"] = ukp["Datetime (Local)"].dt.normalize()


In [161]:
ukp["Hour"] = (
    ukp["Datetime (Local)"]
    .dt.tz_localize("Europe/London", nonexistent="NaT",ambiguous="NaT")
)

In [162]:
ukp_hours = ukp[["Hour", "Price (GBP/MWhe)"]]
ukp_hours[-50:-45]

,Hour,Price (GBP/MWhe)
89950,2026-10-03 22:00:00+01:00,154.963392
89951,2026-10-03 23:00:00+01:00,162.565980
89952,2026-10-04 00:00:00+01:00,159.322322
89953,2026-10-04 01:00:00+01:00,162.438277
89954,2026-10-04 02:00:00+01:00,155.976503


In [163]:
# originally was using a different dataset

'''
ukp = ukp.replace("€/MWh", np.nan)
ukp["Hour"] = pd.to_datetime(
    ukp["Day"].astype(str) + " " + ukp["Hour"].astype(str)
    )
ukp_hours = ukp[["Hour", "Value"]]
ukp_hours["Hour"] = (
    ukp_hours["Hour"]
    .dt.tz_localize("Etc/GMT-1")
    .dt.tz_convert("Europe/London")
)
'''

'\nukp = ukp.replace("€/MWh", np.nan)\nukp["Hour"] = pd.to_datetime(\n    ukp["Day"].astype(str) + " " + ukp["Hour"].astype(str)\n    )\nukp_hours = ukp[["Hour", "Value"]]\nukp_hours["Hour"] = (\n    ukp_hours["Hour"]\n    .dt.tz_localize("Etc/GMT-1")\n    .dt.tz_convert("Europe/London")\n)\n'

In [164]:
day_ahead_demand.head() # UK local time

,DAYSAHEAD,TARGETDATE,FORECASTDEMAND,CARDINALPOINT,CP_TYPE,CP_ST_TIME,CP_END_TIME,F_Point,FORECAST_TIMESTAMP
0,1,2018-01-02,24920,1F,F,30,30,NaN,2018-01-01T08:46:43Z
1,1,2018-01-02,25250,1A,P,100,230,NaN,2018-01-01T08:46:43Z
2,1,2018-01-02,23390,03,F,300,300,NaN,2018-01-01T08:46:43Z
3,1,2018-01-02,22480,1B,T,430,730,Om,2018-01-01T08:46:43Z
4,1,2018-01-02,37190,2F,P,800,900,NaN,2018-01-01T08:46:43Z


In [165]:
# cleaning demand - removing extraneous columns, changing 2400 to 0000 next day

demand = day_ahead_demand[["TARGETDATE", "FORECASTDEMAND", "CP_ST_TIME", "CP_END_TIME", "FORECAST_TIMESTAMP"]]

time_str = demand["CP_ST_TIME"].astype(str).str.zfill(4)
time_str_end = demand["CP_END_TIME"].astype(str).str.zfill(4)

midnight = time_str == "2400"
time_str = time_str.mask(midnight, "0000")
midnight_end = time_str_end == "2400"
time_str_end = time_str_end.mask(midnight_end, "0000")


demand["CP_ST_TIME"] = pd.to_datetime(
    demand["TARGETDATE"].astype(str) + " " +
    time_str.str[:2] + ":" + time_str.str[2:] + ":00")

demand["CP_END_TIME"] = pd.to_datetime(
    demand["TARGETDATE"].astype(str) + " " +
    time_str_end.str[:2] + ":" + time_str_end.str[2:] + ":00")

demand.loc[midnight, "CP_ST_TIME"] += pd.Timedelta(days=1)
demand.loc[midnight_end, "CP_END_TIME"] += pd.Timedelta(days=1)

demand = demand.drop(columns = ["TARGETDATE"])

C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\2757015363.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demand["CP_ST_TIME"] = pd.to_datetime(
C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\2757015363.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demand["CP_END_TIME"] = pd.to_datetime(
C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\2757015363.py:22: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit c

In [166]:
hours = pd.date_range(
    demand["CP_ST_TIME"].min().floor("h"),
    demand["CP_END_TIME"].max().ceil("h"),
    freq="h"
)

start = demand[["CP_ST_TIME", "FORECASTDEMAND"]].rename(
    columns={"CP_ST_TIME": "Endpoint"}
)
end = demand[["CP_END_TIME", "FORECASTDEMAND"]].rename(
    columns={"CP_END_TIME": "Endpoint"}
)

endpoints = pd.concat([start, end], ignore_index=True)
endpoints = endpoints.sort_values("Endpoint")

# Find the closest endpoint for each hour
hourly_df = pd.DataFrame({"Hour": hours})
hourly_df["Endpoint"] = hourly_df["Hour"]

hourly_df = pd.merge_asof(
    hourly_df.sort_values("Hour"),
    endpoints.sort_values("Endpoint"),
    left_on="Hour",
    right_on="Endpoint",
    direction="nearest",
    allow_exact_matches=True
)

In [167]:
demand_hours = hourly_df[["Hour", "FORECASTDEMAND"]]
demand_hours["Hour"] = (
    demand_hours["Hour"]
    .dt.tz_localize("Europe/London", nonexistent="NaT",ambiguous="NaT")
)
demand_hours.head()

C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\1745701569.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demand_hours["Hour"] = (


,Hour,FORECASTDEMAND
0,2018-01-02 00:00:00+00:00,24920
1,2018-01-02 01:00:00+00:00,25250
2,2018-01-02 02:00:00+00:00,25250
3,2018-01-02 03:00:00+00:00,23390
4,2018-01-02 04:00:00+00:00,22480


In [168]:
day_ahead_wind.head() # GMT

,Datetime_GMT,Date,Settlement_period,Capacity,Incentive_forecast,Forecast_Timestamp
0,2018-04-16T23:00:00,2018-04-17,1,11173,7523,2018-04-19T08:05:35
1,2018-04-16T23:30:00,2018-04-17,2,11173,7656,2018-04-19T08:05:35
2,2018-04-17T00:00:00,2018-04-17,3,11173,7736,2018-04-19T08:05:35
3,2018-04-17T00:30:00,2018-04-17,4,11173,7765,2018-04-19T08:05:35
4,2018-04-17T01:00:00,2018-04-17,5,11173,7852,2018-04-19T08:05:35


In [169]:
wind = day_ahead_wind[["Datetime_GMT", "Capacity", "Incentive_forecast"]]
ProfileReport(wind)
;

''

In [170]:
wind["Datetime_GMT"] = pd.to_datetime(wind["Datetime_GMT"])
wind_hours = (
    wind.set_index("Datetime_GMT")
        .resample("1h")["Incentive_forecast"]   
        .mean()
        .reset_index()
)
wind_hours["Hour"] = (
    wind_hours["Datetime_GMT"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Europe/London")
)
wind_hours = wind_hours.drop(columns = ["Datetime_GMT"])

C:\Users\hamis\AppData\Local\Temp\ipykernel_11732\2053493716.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wind["Datetime_GMT"] = pd.to_datetime(wind["Datetime_GMT"])


In [172]:
df = (
    wind_hours
        .merge(demand_hours, on="Hour", how="inner")
        .merge(ukp_hours, on="Hour", how="inner")
)

In [ ]:
ProfileReport(df)
;

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 27.46it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]